In [1]:
!pip install nilearn

In [2]:
from nilearn import datasets
haxby_dataset = datasets.fetch_haxby()
print("Data downloaded!")

[fetch_haxby] Added README.md to /root/nilearn_data

[fetch_haxby] Dataset created in /root/nilearn_data/haxby2001

[fetch_haxby] Downloading data from https://www.nitrc.org/frs/download.php/7868/mask.nii.gz ...

[fetch_haxby]  ...done. (0 seconds, 0 min)

[fetch_haxby] Downloading data from http://data.pymvpa.org/datasets/haxby2001/MD5SUMS ...

...

[fetch_haxby] Downloaded 263962624 of 291168628 bytes (90.7%%, 00 HR 00 MIN 01 SEC remaining)

[fetch_haxby]  ...done. (7 seconds, 0 min)

[fetch_haxby] Extracting data from 
/root/nilearn_data/haxby2001/9cabe068089e791ef0c5fe930fc20e30/subj2-2010.01.14.tar.gz...

[fetch_haxby] .. done.

Data downloaded!


In [12]:
from nilearn.decoding import Decoder
from sklearn.model_selection import LeaveOneGroupOut
import numpy as np

mask_filename = haxby_dataset.mask_vt[0]

# Use run/session numbers for proper cross-validation
groups = labels['chunks'][condition_mask]

logo = LeaveOneGroupOut()

decoder = Decoder(
    estimator='svc',
    mask=mask_filename,
    standardize=True,
    cv=logo
)

decoder.fit(fmri_niimgs, y, groups=groups)

print("Accuracy per run:", decoder.cv_scores_['face'])

mean_accuracy = np.mean(decoder.cv_scores_['face'])
print(f"Mean Leave-One-Run-Out accuracy: {mean_accuracy * 100:.2f}%")


/tmp/ipykernel_8086/3359559423.py:19: UserWarning: The provided image has no sform in its header. Please check the provided file. Results may not be as expected.
  decoder.fit(fmri_niimgs, y, groups=groups)
/tmp/ipykernel_8086/3359559423.py:19: UserWarning: screening_percentile set to '100' despite requesting 'screening_percentile=20'. 
All elements in the mask will be included. 
This usually occurs when the mask image is too small compared to full brain mask.
  decoder.fit(fmri_niimgs, y, groups=groups)


Accuracy per run: [np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0)]
Mean Leave-One-Run-Out accuracy: 100.00%


In [4]:
print(haxby_dataset.description)

.. _haxby_dataset:

Haxby dataset

Access
------
See :func:`nilearn.datasets.fetch_haxby`.

Notes
-----
Results from a classical :term:`fMRI` study that investigated the differences between
the neural correlates of face versus object processing in the ventral visual
stream. Face and object stimuli showed widely distributed and overlapping
response patterns.

See :footcite:t:`Haxby2001`.

Content
-------
The "simple" dataset includes:
    :'func': Nifti images with bold data
    :'session_target': Text file containing run data
    :'mask': Nifti images with employed mask
    :'session': Text file with condition labels

The full dataset additionally includes
    :'anat': Nifti images with anatomical image
    :'func': Nifti images with bold data
    :'mask_vt': Nifti images with mask for ventral visual/temporal cortex
    :'mask_face': Nifti images with face-reponsive brain regions
    :'mask_house': Nifti images with house-reponsive brain regions
    :'mask_face_little': Spatially more 

In [5]:
import pandas as pd

# Load the labels (what the subject was looking at during each scan)
labels = pd.read_csv(haxby_dataset.session_target[0], sep=" ")
print(labels['labels'].unique())

<ArrowStringArray>
[        'rest',     'scissors',         'face',          'cat',
         'shoe',        'house', 'scrambledpix',       'bottle',
        'chair']
Length: 9, dtype: str


In [6]:
condition_mask = labels['labels'].isin(['face', 'house'])
y = labels['labels'][condition_mask]
print(y.value_counts())

labels
face     108
house    108
Name: count, dtype: int64


In [7]:
from nilearn.image import index_img

func_filename = haxby_dataset.func[0]
fmri_niimgs = index_img(func_filename, condition_mask)
print(fmri_niimgs.shape)

(40, 64, 64, 216)


In [9]:
from nilearn.decoding import Decoder
from sklearn.model_selection import cross_val_score, KFold

mask_filename = haxby_dataset.mask_vt[0]

cv = KFold(n_splits=5)
decoder = Decoder(estimator='svc', mask=mask_filename, standardize=True, cv=cv)
decoder.fit(fmri_niimgs, y)

print("Cross-validated accuracy per fold:", decoder.cv_scores_['face'])

import numpy as np
mean_accuracy = np.mean(decoder.cv_scores_['face'])
print(f"Mean cross-validated accuracy: {mean_accuracy * 100:.2f}%")

/tmp/ipykernel_8086/1973507516.py:8: UserWarning: The provided image has no sform in its header. Please check the provided file. Results may not be as expected.
  decoder.fit(fmri_niimgs, y)


Cross-validated accuracy per fold: [np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0)]
Mean cross-validated accuracy: 100.00%


/tmp/ipykernel_8086/1973507516.py:8: UserWarning: screening_percentile set to '100' despite requesting 'screening_percentile=20'. 
All elements in the mask will be included. 
This usually occurs when the mask image is too small compared to full brain mask.
  decoder.fit(fmri_niimgs, y)


In [11]:
print(labels.columns)

Index(['labels', 'chunks'], dtype='str')


In [13]:
print(decoder.cv_scores_)


{np.str_('face'): [np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0)], np.str_('house'): [np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0)]}
